In [10]:
!git clone https://github.com/Priya-Kumari-Chourasia/word_embed

fatal: destination path 'word_embed' already exists and is not an empty directory.


In [11]:
import torch
import random
import os
# loadinf the data
# rading the names from dataset
data_path = os.path.join("word_embed", "TrainingNames.txt")

with open(data_path, "r", encoding="utf-8") as f:
    names = f.read().splitlines()

# conevrting all names to lowecase
names = [n.lower() for n in names]

#names = [n.split()[0] for n in names]


print(names[:5])

# extracting all unique characters from dataset
chars = sorted(list(set("".join(names))))

# adding tokens for start and end
chars = ["<s>","</s>"] + chars

stoi = {ch:i for i,ch in enumerate(chars)}
itos = {i:ch for ch,i in stoi.items()}

vocab_size = len(chars)

print(chars)

# encoding function
# converting a name into sequence of indices
def encode(name):

    seq = [stoi["<s>"]]

    for c in name:
        seq.append(stoi[c])

    seq.append(stoi["</s>"])

    return seq

# preparing the dataset as (inpt ,target)pairs
# target means next char to be predicted
dataset = []

for name in names:

    seq = encode(name)

    x = seq[:-1]
    y = seq[1:]

    dataset.append((x,y))

for i in range(5):
    print(dataset[i])


#creating random mini  batches
# since diff names have diff length so paddeing is required
def bat(batch_size=32):

    X = []
    Y = []

    for _ in range(batch_size):

        x, y = random.choice(dataset)

        X.append(x)
        Y.append(y)

    max_len = max(len(x) for x in X)

    Xpad = []
    Ypad = []

    for x,y in zip(X,Y):

        Xpad.append(x + [0]*(max_len-len(x)))
        Ypad.append(y + [0]*(max_len-len(y)))

    return torch.tensor(Xpad), torch.tensor(Ypad)


# here implementing rnn from scratch

class rnn_scratch(torch.nn.Module):

    def __init__(self,num_inputs,num_hiddens):

        super().__init__()

        self.num_hiddens = num_hiddens

        self.W_xh = torch.nn.Parameter(
            torch.randn(num_inputs,num_hiddens)*0.01)

        self.W_hh = torch.nn.Parameter(
            torch.randn(num_hiddens,num_hiddens)*0.01)

        self.b_h = torch.nn.Parameter(torch.zeros(num_hiddens))

    def forward(self,inputs,state):

        outputs = []

        for X in inputs:

            state = torch.tanh(
                X @ self.W_xh + state @ self.W_hh + self.b_h
            )

            outputs.append(state)

        return outputs,state

# using rnn to predict next character
class rnn(torch.nn.Module):

    def __init__(self,rnn,vocab_size):

        super().__init__()

        self.rnn = rnn

        self.vocab_size = vocab_size

        self.W_hq = torch.nn.Parameter(
            torch.randn(rnn.num_hiddens,vocab_size)*0.01)

        self.b_q = torch.nn.Parameter(torch.zeros(vocab_size))


    def forward(self,X):

        X = torch.nn.functional.one_hot(
            X.T,self.vocab_size).float()

        state = torch.zeros(
            (X.shape[1],self.rnn.num_hiddens))

        rnn_outputs,_ = self.rnn(X,state)

        outputs = []

        for H in rnn_outputs:

            Y = H @ self.W_hq + self.b_q

            outputs.append(Y)

        return torch.stack(outputs,1)


model = rnn(
    rnn_scratch(vocab_size,128),
    vocab_size
)

optimizer = torch.optim.Adam(model.parameters(),lr=0.003)

loss_fn = torch.nn.CrossEntropyLoss(ignore_index=0)

for epoch in range(500):

    X,Y = bat()

    pred = model(X)

    loss = loss_fn(
        pred.reshape(-1,vocab_size),
        Y.reshape(-1)
    )

    optimizer.zero_grad()

    loss.backward()

    optimizer.step()

    if epoch%20==0:
        print(epoch,loss.item())


def gen_rnn():

    name = ["<s>"]

    for _ in range(25):

        seq = torch.tensor([[stoi[c] for c in name]], dtype=torch.long)

        out = model(seq)

        temperature = 0.8
        probs = torch.softmax(out[0,-1] / temperature, dim=0)

        idx = torch.multinomial(probs,1).item()

        char = itos[idx]

        if char == "</s>":
            break

        name.append(char)

    return "".join(name[1:])


for _ in range(10):
    print(gen_rnn())



['ashish walia', 'rakesh vaidya', 'hemant sen', 'siddharth adiga', 'vihaan bansal']
['<s>', '</s>', ' ', 'a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k', 'l', 'm', 'n', 'o', 'p', 'r', 's', 't', 'u', 'v', 'w', 'x', 'y', 'z']
([0, 3, 20, 10, 11, 20, 10, 2, 24, 3, 14, 11, 3], [3, 20, 10, 11, 20, 10, 2, 24, 3, 14, 11, 3, 1])
([0, 19, 3, 13, 7, 20, 10, 2, 23, 3, 11, 6, 26, 3], [19, 3, 13, 7, 20, 10, 2, 23, 3, 11, 6, 26, 3, 1])
([0, 10, 7, 15, 3, 16, 21, 2, 20, 7, 16], [10, 7, 15, 3, 16, 21, 2, 20, 7, 16, 1])
([0, 20, 11, 6, 6, 10, 3, 19, 21, 10, 2, 3, 6, 11, 9, 3], [20, 11, 6, 6, 10, 3, 19, 21, 10, 2, 3, 6, 11, 9, 3, 1])
([0, 23, 11, 10, 3, 3, 16, 2, 4, 3, 16, 20, 3, 14], [23, 11, 10, 3, 3, 16, 2, 4, 3, 16, 20, 3, 14, 1])
0 3.3322696685791016
20 2.928004503250122
40 2.845543146133423
60 2.7091190814971924
80 2.539337635040283
100 2.393355369567871
120 2.318791627883911
140 2.1370131969451904
160 2.043395757675171
180 2.0208137035369873
200 1.9453014135360718
220 1.9385100603103638
240

In [ ]:
import torch
import random

#reading the generated names from dataset
data_path = os.path.join("word_embed", "TrainingNames.txt")

with open(data_path, "r", encoding="utf-8") as f:
    names = f.read().splitlines()




names = [n.lower() for n in names]

#  first i have tried using only first name but here i have kept full name

#names = [n.split()[0] for n in names]


print(names[:5])

chars = ["<pad>", "<s>", "</s>"] + sorted(list(set("".join(names))))

stoi = {ch:i for i,ch in enumerate(chars)}
itos = {i:ch for ch,i in stoi.items()}

# here i have taken cross entropy loss
ignore_index = stoi["<pad>"]
loss_fn = torch.nn.CrossEntropyLoss(ignore_index=ignore_index)



vocab_size = len(chars)

print(chars)

# encoding the names
# adding <s> at start and </s> at end so model learns boundaries

def encode(name):

    seq = [stoi["<s>"]]

    for c in name:
        seq.append(stoi[c])

    seq.append(stoi["</s>"])

    return seq


dataset = []

for name in names:

    seq = encode(name)

    x = seq[:-1]
    y = seq[1:]

    dataset.append((x,y))

for i in range(5):
    print(dataset[i])


# function for batch sampling
# it is randomly picking names and pads them to dame length
def bat(batch_size=32):

    X = []
    Y = []

    for _ in range(batch_size):

        x, y = random.choice(dataset)

        X.append(x)
        Y.append(y)

    max_len = max(len(x) for x in X)

    Xpad = []
    Ypad = []

    for x,y in zip(X,Y):

        Xpad.append(x + [stoi["<pad>"]]*(max_len-len(x)))
        Ypad.append(y + [stoi["<pad>"]]*(max_len-len(y)))
        #Xpad.append(x + [stoi["<pad>"]] * (max_len-len(x)))
        #Ypad.append(y + [stoi["<pad>"]] * (max_len-len(y)))

    return torch.tensor(Xpad), torch.tensor(Ypad)

# implementing the basic LSTM fromscatch without using any module
class lstm_scr(torch.nn.Module):

    def __init__(self, input_size, hidden_size):

        super().__init__()

        self.hidden_size = hidden_size

        def param(shape):
            return torch.nn.Parameter(torch.randn(shape)*0.01)

        # input gate
        self.W_xi = param((input_size, hidden_size))
        self.W_hi = param((hidden_size, hidden_size))
        self.b_i = torch.nn.Parameter(torch.zeros(hidden_size))

        # forget gate
        self.W_xf = param((input_size, hidden_size))
        self.W_hf = param((hidden_size, hidden_size))
        self.b_f = torch.nn.Parameter(torch.zeros(hidden_size))

        # output gate
        self.W_xo = param((input_size, hidden_size))
        self.W_ho = param((hidden_size, hidden_size))
        self.b_o = torch.nn.Parameter(torch.zeros(hidden_size))

        # candidate memory
        self.W_xc = param((input_size, hidden_size))
        self.W_hc = param((hidden_size, hidden_size))
        self.b_c = torch.nn.Parameter(torch.zeros(hidden_size))


    def forward(self, inputs, state):

        H, C = state
        outputs = []

        for X in inputs:

            I = torch.sigmoid(X @ self.W_xi + H @ self.W_hi + self.b_i)
            F = torch.sigmoid(X @ self.W_xf + H @ self.W_hf + self.b_f)
            O = torch.sigmoid(X @ self.W_xo + H @ self.W_ho + self.b_o)

            C_tilde = torch.tanh(X @ self.W_xc + H @ self.W_hc + self.b_c)

            C = F*C + I*C_tilde
            H = O*torch.tanh(C)

            outputs.append(H)

        return outputs,(H,C)

# running lstm both in forward and backward and combines the outputs
class bi_lstm_scr(torch.nn.Module):

    def __init__(self,input_size,hidden_size):

        super().__init__()

        self.hidden_size = hidden_size

        self.forward_lstm = lstm_scr(input_size,hidden_size)
        self.backward_lstm = lstm_scr(input_size,hidden_size)


    def forward(self,inputs,state=None):

        batch_size = inputs.shape[1]

        Hf = torch.zeros((batch_size,self.hidden_size))
        Cf = torch.zeros((batch_size,self.hidden_size))

        Hb = torch.zeros((batch_size,self.hidden_size))
        Cb = torch.zeros((batch_size,self.hidden_size))

        forward_out,_ = self.forward_lstm(inputs,(Hf,Cf))

        backward_inputs = list(reversed(inputs))
        backward_out,_ = self.backward_lstm(backward_inputs,(Hb,Cb))

        backward_out = list(reversed(backward_out))

        outputs = []

        for f,b in zip(forward_out,backward_out):

            outputs.append(torch.cat((f,b),dim=1))

        return outputs

# it is bilstm language model
class bilstm(torch.nn.Module):

    def __init__(self, vocab_size, hidden_size=256, embed_size=32):

        super().__init__()

        self.embedding = torch.nn.Embedding(vocab_size, embed_size)

        self.blstm = bi_lstm_scr(embed_size, hidden_size)

        self.dropout = torch.nn.Dropout(0.1)

        self.vocab_size = vocab_size

        self.W_hq = torch.nn.Parameter(
            torch.randn(hidden_size*2, vocab_size)*0.01)

        self.b_q = torch.nn.Parameter(torch.zeros(vocab_size))


    def forward(self,X):

        X = self.embedding(X)   # (batch, seq, embed)

        X = X.permute(1,0,2)    # (seq, batch, embed)

        outputs = self.blstm(X)

        Y = []

        for H in outputs:
            H = self.dropout(H)  # Add this
            Y.append(H @ self.W_hq + self.b_q)

        return torch.stack(Y,1)

# here traing the model

model = bilstm(vocab_size, hidden_size=128, embed_size=32)

optimizer = torch.optim.Adam(model.parameters(),lr=0.003)

loss_fn = torch.nn.CrossEntropyLoss(ignore_index=0)
#loss_fn = torch.nn.CrossEntropyLoss(ignore_index=stoi["<pad>"])

for epoch in range(500):

    X,Y = bat()

    pred = model(X)

    loss = loss_fn(
        pred.reshape(-1,vocab_size),
        Y.reshape(-1)
    )

    optimizer.zero_grad()

    loss.backward()

    optimizer.step()

    if epoch%50==0:
        print(epoch,loss.item())

# generating the names
def gen_bilstm():

    name = ["<s>"]

    for _ in range(20):

        seq = torch.tensor([[stoi[c] for c in name]], dtype=torch.long)

        out = model(seq)

        temperature = 1.0
        probs = torch.softmax(out[0,-1] / temperature, dim=0)

        # prevent <s> appearing again
        probs[stoi["<s>"]] = 0
        probs[stoi["</s>"]] = probs[stoi["</s>"]] * 0.5
        probs = probs / probs.sum()

        idx = torch.multinomial(probs,1).item()

        char = itos[idx]

        if char == "</s>" and len(name) < 6:
            continue

        if char == "</s>":
            break

        name.append(char)

    return "".join(name[1:])


for _ in range(10):
    print(gen_bilstm())

['ashish walia', 'rakesh vaidya', 'hemant sen', 'siddharth adiga', 'vihaan bansal']
['<pad>', '<s>', '</s>', ' ', 'a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k', 'l', 'm', 'n', 'o', 'p', 'r', 's', 't', 'u', 'v', 'w', 'x', 'y', 'z']
([1, 4, 21, 11, 12, 21, 11, 3, 25, 4, 15, 12, 4], [4, 21, 11, 12, 21, 11, 3, 25, 4, 15, 12, 4, 2])
([1, 20, 4, 14, 8, 21, 11, 3, 24, 4, 12, 7, 27, 4], [20, 4, 14, 8, 21, 11, 3, 24, 4, 12, 7, 27, 4, 2])
([1, 11, 8, 16, 4, 17, 22, 3, 21, 8, 17], [11, 8, 16, 4, 17, 22, 3, 21, 8, 17, 2])
([1, 21, 12, 7, 7, 11, 4, 20, 22, 11, 3, 4, 7, 12, 10, 4], [21, 12, 7, 7, 11, 4, 20, 22, 11, 3, 4, 7, 12, 10, 4, 2])
([1, 24, 12, 11, 4, 4, 17, 3, 5, 4, 17, 21, 4, 15], [24, 12, 11, 4, 4, 17, 3, 5, 4, 17, 21, 4, 15, 2])
0 3.3675575256347656
50 0.9228197336196899
100 0.10524464398622513
150 0.02081068977713585
200 0.012379512190818787
250 0.006879165302962065
300 0.005334597546607256
350 0.003833082504570484
400 0.0031622659880667925


In [ ]:
import torch
import random

# loading the names from the dataset where each line corresponds to one name
data_path = os.path.join("word_embed", "TrainingNames.txt")

with open(data_path, "r", encoding="utf-8") as f:
    names = f.read().splitlines()


# converting all the names to lowecase
names = [n.lower() for n in names]

chars = sorted(list(set("".join(names))))
chars = ["<pad>","<s>","</s>"] + chars

# it is mapping from index to char and char to index
stoi = {ch:i for i,ch in enumerate(chars)}
itos = {i:ch for ch,i in stoi.items()}

vocab_size = len(chars)


# encoding the dataset
# here adding the start and end token so that model learn where to start and stop
def encode(name):

    seq = [stoi["<s>"]]

    for c in name:
        seq.append(stoi[c])

    seq.append(stoi["</s>"])

    return seq


dataset = []

for name in names:

    seq = encode(name)

    x = seq[:-1]
    y = seq[1:]

    dataset.append((x,y))


# function for the batch loader
def bat(batch_size=32):

    X,Y = [],[]

    for _ in range(batch_size):

        x,y = random.choice(dataset)

        X.append(x)
        Y.append(y)

    max_len = max(len(x) for x in X)

    Xpad,Ypad = [],[]

    for x,y in zip(X,Y):

        Xpad.append(x + [stoi["<pad>"]] * (max_len-len(x)))
        Ypad.append(y + [stoi["<pad>"]] * (max_len-len(y)))

    return torch.tensor(Xpad), torch.tensor(Ypad)


# this is the function for RNN cell
# it is coded froom scratch which captures sequntial dependencies in char
class scr_Rnn(torch.nn.Module):

    def __init__(self,input_size,hidden_size):

        super().__init__()

        self.hidden_size = hidden_size

        self.W_xh = torch.nn.Parameter(
            torch.randn(input_size,hidden_size)*0.01)

        self.W_hh = torch.nn.Parameter(
            torch.randn(hidden_size,hidden_size)*0.01)

        self.b_h = torch.nn.Parameter(torch.zeros(hidden_size))


    def forward(self,X,H):

        H = torch.tanh(X @ self.W_xh + H @ self.W_hh + self.b_h)

        return H


# function for RNN + Attention model
# it is improving basic rnn by adding attention mechanism
class rnn_att(torch.nn.Module):

    def __init__(self,vocab_size,hidden_size=128,embed_size=32):

        super().__init__()

        self.hidden_size = hidden_size

        self.embedding = torch.nn.Embedding(vocab_size,embed_size)

        self.rnn = scr_Rnn(embed_size,hidden_size)

        # attention weights
        self.W_attn = torch.nn.Parameter(
            torch.randn(hidden_size,hidden_size)*0.01)

        # output layer
        self.W_out = torch.nn.Parameter(
            torch.randn(hidden_size*2,vocab_size)*0.01)

        self.b_out = torch.nn.Parameter(torch.zeros(vocab_size))


    def forward(self,X):

        X = self.embedding(X)

        X = X.permute(1,0,2)

        batch_size = X.shape[1]

        H = torch.zeros(batch_size,self.hidden_size)

        hidden_states = []

        outputs = []

        for t in range(X.shape[0]):

            H = self.rnn(X[t],H)

            hidden_states.append(H)

            hs = torch.stack(hidden_states)          # (t+1, batch, hidden)

            hs = hs.permute(1,0,2)                   # (batch, t+1, hidden)

            scores = torch.matmul(
                torch.matmul(hs, self.W_attn),
                H.unsqueeze(2)
            )                                        # (batch, t+1, 1)

            scores = scores.squeeze(2)               # (batch, t+1)

            attn_weights = torch.softmax(scores,dim=1)

            context = torch.sum(
                attn_weights.unsqueeze(2) * hs,
                dim=1
            )                                        # (batch, hidden)

            combined = torch.cat((H,context),dim=1)

            Y = combined @ self.W_out + self.b_out

            outputs.append(Y)

        return torch.stack(outputs,1)


# model function
model = rnn_att(vocab_size)

#using adam optimizer for training
optimizer = torch.optim.Adam(model.parameters(),lr=0.003)

# ignoring tha padding tokens while computing loss
loss_fn = torch.nn.CrossEntropyLoss(ignore_index=stoi["<pad>"])


# training the data
for epoch in range(500):

    X,Y = bat()

    pred = model(X)

    loss = loss_fn(
        pred.reshape(-1,vocab_size),
        Y.reshape(-1)
    )

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    if epoch%50==0:
        print(epoch,loss.item())


# this function is generating the names
def gen_attn():

    name = ["<s>"]

    for _ in range(30):

        seq = torch.tensor([[stoi[c] for c in name]])

        out = model(seq)

        temperature = 1.0

        probs = torch.softmax(out[0,-1]/temperature,dim=0)

        probs[stoi["<s>"]] = 0
        probs[stoi["<pad>"]] = 0

        probs = probs / probs.sum()

        idx = torch.multinomial(probs,1).item()

        char = itos[idx]

        if char == "</s>" and len(name) < 6:
            continue

        if char == "</s>":
            break

        name.append(char)

    return "".join(name[1:])


# priting the 10 generated names
for _ in range(10):
    print(gen_attn())

In [ ]:
def evaluate(gen_function, train_names, num_samples=500):

    generated = []

    for _ in range(num_samples):
        generated.append(gen_function())

    train_set = set(train_names)

    # Novelty: names not in training set
    novel = [n for n in generated if n not in train_set]
    novelty_rate = len(novel) / num_samples * 100

    # Diversity: unique names
    diversity = len(set(generated)) / num_samples

    return novelty_rate, diversity



# RNN
rnn_nov, rnn_div = evaluate(gen_rnn, names)

# BiLSTM
bilstm_nov, bilstm_div = evaluate(gen_bilstm, names)

# Attention RNN
attn_nov, attn_div = evaluate(gen_attn, names)

# Print table
print("\n===== Quantitative Evaluation =====")
print(f"{'Model':<20}{'Novelty (%)':<15}{'Diversity':<10}")
print("-"*45)
print(f"{'Vanilla RNN':<20}{rnn_nov:<15.2f}{rnn_div:<10.2f}")
print(f"{'BiLSTM':<20}{bilstm_nov:<15.2f}{bilstm_div:<10.2f}")
print(f"{'RNN + Attention':<20}{attn_nov:<15.2f}{attn_div:<10.2f}")